# Man vs Machine on Oslo Bors - Evaluation

### Data setup

In [24]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Loading and aligning datasets...")

# 1. Load your Simulation Returns (from Notebook 3)
sim_returns_mod = pd.read_csv('../Data/portfolio_simulation_returns_mod.csv')
sim_returns_mod['TradeDate'] = pd.to_datetime(sim_returns_mod['TradeDate']) + pd.offsets.MonthEnd(0)

sim_returns_hist = pd.read_csv('../Data/portfolio_simulation_returns_hist.csv')
sim_returns_hist['TradeDate'] = pd.to_datetime(sim_returns_hist['TradeDate']) + pd.offsets.MonthEnd(0)

# 2. Load Ødegaard's Risk-Free Rate (Note: skiprows=1 to bypass the text header)
rf_df = pd.read_csv('../Data/Norway_Rf_monthly.csv', skiprows=1)
rf_df['TradeDate'] = pd.to_datetime(rf_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
rf_df = rf_df[['TradeDate', 'Rf(1m)']]

# 3. Load Ødegaard's Market Portfolios (We specifically want 'VW')
mkt_df = pd.read_csv('../Data/Norway_market_portfolios_monthly.csv')
mkt_df['TradeDate'] = pd.to_datetime(mkt_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
mkt_df = mkt_df[['TradeDate', 'VW']] 

# 4. Load Ødegaard's Fama-French Factors (SMB, HML, UMD)
ff_df = pd.read_csv('../Data/Norway_pricing_factors_monthly.csv')
ff_df['TradeDate'] = pd.to_datetime(ff_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
ff_df = ff_df[['TradeDate', 'SMB', 'HML', 'UMD']]

# 5. Master Merge
eval_df_mod = sim_returns_mod.merge(rf_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(mkt_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(ff_df, on='TradeDate', how='left')

eval_df_hist = sim_returns_hist.merge(rf_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(mkt_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(ff_df, on='TradeDate', how='left')

# Drop any rows where Ødegaard's data might be missing at the very end of our sample
eval_df_mod = eval_df_mod.dropna().reset_index(drop=True)
eval_df_hist = eval_df_hist.dropna().reset_index(drop=True)


# 6. Calculate the Market Risk Premium (Rm - Rf)
eval_df_mod['Mkt-RF'] = eval_df_mod['VW'] - eval_df_mod['Rf(1m)']
eval_df_hist['Mkt-RF'] = eval_df_hist['VW'] - eval_df_hist['Rf(1m)']

# 6.5 add momentum column to the historical dataframe
eval_df_hist['Net_Ret_Mom_LS'] = eval_df_mod['Net_Ret_Mom_LS']

# 7. Define the specific strategies we want to evaluate
# This list will make looping through the risk metrics and regressions much cleaner!
strategy_cols_mod = [
    'Net_Ret_LongOnly',       # The Mutual Fund AI strategy
    'Net_Ret_LongShort',      # The Hedge Fund AI strategy (Ensemble)
    'Net_Ret_Modern_RF_LS',   # The Random Forest Hedge Fund strategy
    'Net_Ret_Mom_LS',         # The Traditional Finance baseline
    'VW'                      # The overall Market
]

strategy_cols_hist = [
    'Net_Ret_LongOnly',       # The Mutual Fund AI strategy
    'Net_Ret_LongShort',      # The Hedge Fund AI strategy (Ensemble)
    'Net_Ret_Hist_RF_LS',   # The Random Forest Hedge Fund strategy
    'Net_Ret_Mom_LS',         # The Traditional Finance baseline
    'VW'                      # The overall Market
]

print(f"Data successfully aligned! Total valid months: {eval_df_mod.shape[0]}")
print("\nPreview of the Evaluation Matrix:")
display(eval_df_mod[['TradeDate'] + strategy_cols_mod + ['Rf(1m)', 'Mkt-RF', 'SMB', 'HML', 'UMD']].head())

Loading and aligning datasets...
Data successfully aligned! Total valid months: 34

Preview of the Evaluation Matrix:


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Modern_RF_LS,Net_Ret_Mom_LS,VW,Rf(1m),Mkt-RF,SMB,HML,UMD
0,2018-01-31,-0.010172,0.011868,-0.000723,-0.000045,0.000890,0.00063,0.000260,0.009561,0.018891,-0.001068
1,2018-02-28,-0.030890,0.009704,0.002631,0.025318,0.008938,0.00078,0.008158,-0.001476,-0.014693,-0.051646
2,2018-03-31,0.055877,-0.005287,0.003752,-0.067810,-0.018797,0.00079,-0.019587,0.022721,-0.033547,-0.002256
3,2018-04-30,0.032707,-0.043988,-0.052799,-0.026148,0.068472,0.00076,0.067712,-0.022607,-0.013137,-0.054883
4,2018-05-31,-0.011356,0.037239,0.060911,-0.053774,0.025658,0.00068,0.024978,0.021633,0.016803,-0.041950


In [25]:
display(eval_df_hist)
display(eval_df_mod)

,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Hist_RF_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF,Net_Ret_Mom_LS
0,2018-01-31,0.003672,0.035543,0.007862,1.000000,2.000000,0.029034,0.00063,0.000890,0.009561,0.018891,-0.001068,0.000260,-0.000045
1,2018-02-28,-0.033991,0.010935,-0.028993,0.722222,1.500000,0.003281,0.00078,0.008938,-0.001476,-0.014693,-0.051646,0.008158,0.025318
2,2018-03-31,0.032729,-0.072316,0.059075,0.685714,1.485714,-0.011471,0.00079,-0.018797,0.022721,-0.033547,-0.002256,-0.019587,-0.067810
3,2018-04-30,0.013318,-0.057404,0.031829,0.857143,1.542857,-0.065617,0.00076,0.068472,-0.022607,-0.013137,-0.054883,0.067712,-0.026148
4,2018-05-31,-0.013313,0.039520,-0.007192,0.857143,1.485714,0.077251,0.00068,0.025658,0.021633,0.016803,-0.041950,0.024978,-0.053774
5,2018-06-30,0.024608,0.044375,0.013052,0.777778,1.500000,0.054201,0.00064,0.006066,-0.012500,-0.006324,-0.018179,0.005426,0.022506
6,2018-07-31,0.006280,0.081996,-0.005280,0.666667,1.166667,0.088510,0.00065,0.011956,0.036330,-0.087618,0.019781,0.011306,0.064246
7,2018-08-31,0.024548,0.011491,0.011007,0.702703,1.405405,0.008131,0.00068,0.011248,0.020002,0.008085,-0.026814,0.010568,0.000872
8,2018-09-30,-0.046834,0.070994,-0.058922,0.864865,1.405405,0.047902,0.00082,0.037830,0.022083,-0.046851,0.013125,0.037010,-0.017585
9,2018-10-31,-0.083607,-0.058490,-0.049400,0.833333,1.722222,-0.065853,0.00082,-0.057721,-0.037511,0.055993,0.005661,-0.058541,-0.061763


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Modern_RF_LS,Net_Ret_Mom_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF
0,2018-01-31,-0.010172,0.011868,0.007862,1.000000,2.000000,-0.000723,-0.000045,0.00063,0.000890,0.009561,0.018891,-0.001068,0.000260
1,2018-02-28,-0.030890,0.009704,-0.028993,0.888889,1.833333,0.002631,0.025318,0.00078,0.008938,-0.001476,-0.014693,-0.051646,0.008158
2,2018-03-31,0.055877,-0.005287,0.059075,0.914286,1.600000,0.003752,-0.067810,0.00079,-0.018797,0.022721,-0.033547,-0.002256,-0.019587
3,2018-04-30,0.032707,-0.043988,0.031829,0.742857,1.314286,-0.052799,-0.026148,0.00076,0.068472,-0.022607,-0.013137,-0.054883,0.067712
4,2018-05-31,-0.011356,0.037239,-0.007192,0.742857,1.485714,0.060911,-0.053774,0.00068,0.025658,0.021633,0.016803,-0.041950,0.024978
5,2018-06-30,0.023099,0.021224,0.013052,0.833333,1.722222,0.006998,0.022506,0.00064,0.006066,-0.012500,-0.006324,-0.018179,0.005426
6,2018-07-31,0.052986,0.105629,-0.005280,0.833333,1.555556,0.101552,0.064246,0.00065,0.011956,0.036330,-0.087618,0.019781,0.011306
7,2018-08-31,0.024578,-0.000176,0.011007,0.918919,1.621622,-0.022079,0.000872,0.00068,0.011248,0.020002,0.008085,-0.026814,0.010568
8,2018-09-30,-0.071578,0.025978,-0.058922,0.810811,1.675676,0.046456,-0.017585,0.00082,0.037830,0.022083,-0.046851,0.013125,0.037010
9,2018-10-31,-0.030723,0.028241,-0.049400,0.666667,1.444444,0.012628,-0.061763,0.00082,-0.057721,-0.037511,0.055993,0.005661,-0.058541


### Modern evaluation

In [26]:
# ==========================================
# CELL 2: RISK METRICS (SHARPE & DRAWDOWN)
# ==========================================
import pandas as pd
import numpy as np

print("Calculating Risk Metrics...\n")

risk_results = []

for col in strategy_cols_mod:
    # 1. Calculate Excess Return for the strategy
    # Note: If the strategy is 'VW', it is already the market, but we still subtract Rf for its Sharpe
    excess_return = eval_df_mod[col] - eval_df_mod['Rf(1m)']
    
    # 2. Calculate Annualized Return and Volatility
    ann_ret = eval_df_mod[col].mean() * 12
    ann_vol = eval_df_mod[col].std() * np.sqrt(12)
    
    # 3. Calculate Annualized Sharpe Ratio
    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0
    
    # 4. Calculate Maximum Drawdown
    # Create a cumulative wealth index starting at 1.0
    cum_wealth = (1 + eval_df_mod[col]).cumprod()
    # Track the highest peak achieved so far
    running_max = cum_wealth.cummax()
    # Calculate the percentage drop from the peak
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()
    
    # Save the metrics
    risk_results.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

# Convert to a DataFrame for clean formatting
risk_df = pd.DataFrame(risk_results)

# Format the output beautifully for your thesis
risk_df['Ann_Return'] = risk_df['Ann_Return'].map('{:.2%}'.format)
risk_df['Ann_Volatility'] = risk_df['Ann_Volatility'].map('{:.2%}'.format)
risk_df['Sharpe_Ratio'] = risk_df['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df['Max_Drawdown'] = risk_df['Max_Drawdown'].map('{:.2%}'.format)

print("=== RISK-ADJUSTED PERFORMANCE METRICS ===")
print(risk_df)

Calculating Risk Metrics...

=== RISK-ADJUSTED PERFORMANCE METRICS ===
               Strategy Ann_Return Ann_Volatility Sharpe_Ratio Max_Drawdown
0      Net_Ret_LongOnly     24.04%         20.47%         1.13      -22.27%
1     Net_Ret_LongShort     26.78%         30.02%         0.86      -26.55%
2  Net_Ret_Modern_RF_LS     20.77%         25.04%         0.79      -23.46%
3        Net_Ret_Mom_LS     15.39%         29.56%         0.49      -24.05%
4                    VW     -0.19%         16.18%        -0.07      -27.04%


In [27]:
# ==========================================
# CELL 3: THE ULTIMATE ALPHA REGRESSIONS
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Running CAPM, FF3, and Carhart 4-Factor Regressions...\n")

# We don't need to run this on the VW market itself, just our strategies
strategies_to_test = [
    'Net_Ret_LongOnly', 
    'Net_Ret_LongShort', 
    'Net_Ret_Modern_RF_LS', 
    'Net_Ret_Mom_LS'
]

# Create an empty list to store our rows of data
alpha_results = []

for strat in strategies_to_test:
    # Our Dependent Variable (Y) is the Strategy's Excess Return
    Y = eval_df_mod[strat] - eval_df_mod['Rf(1m)']
    
    # ----------------------------------------------------
    # Model 1: CAPM (1-Factor)
    # ----------------------------------------------------
    X_capm = sm.add_constant(eval_df_mod[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()
    
    # ----------------------------------------------------
    # Model 2: Fama-French 3-Factor (FF3)
    # ----------------------------------------------------
    X_ff3 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()
    
    # ----------------------------------------------------
    # Model 3: Carhart 4-Factor (FF4)
    # ----------------------------------------------------
    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # Extract Alpha (Intercept) and annualize it (* 12)
    # We also extract the t-stat, p-value, and R-squared
    alpha_results.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],
        
        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],
        
        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

# Format the results into a beautiful Pandas DataFrame
alpha_df = pd.DataFrame(alpha_results)

# Create a mapping function to add significance stars based on p-values
def format_alpha(alpha, pval):
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f"{alpha:.2%}{stars}"

# Apply formatting
for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df[f'{model}_Alpha_Str'] = alpha_df.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df[f'{model}_t'] = alpha_df[f'{model}_t'].map('{:.2f}'.format)
    alpha_df[f'{model}_p'] = alpha_df[f'{model}_p'].map('{:.3f}'.format)

# Reorder columns to match your requested table format perfectly
final_table = alpha_df[[
    'Strategy', 
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]]

# Rename for display
final_table.columns = [
    'Strategy', 
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha', 
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat', 
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== COMBINED ALPHA SUMMARY TABLE ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(final_table)

Running CAPM, FF3, and Carhart 4-Factor Regressions...

=== COMBINED ALPHA SUMMARY TABLE ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Net_Ret_LongOnly,23.18%*,4.38%,5.54%,1.88,0.33,0.41,0.069,0.746,0.687
1,Net_Ret_LongShort,26.41%,49.65%**,45.43%**,1.52,2.55,2.42,0.139,0.016,0.022
2,Net_Ret_Modern_RF_LS,20.18%,29.76%*,27.21%,1.37,1.71,1.57,0.181,0.097,0.127
3,Net_Ret_Mom_LS,14.85%,23.93%,21.01%,0.85,1.15,1.01,0.402,0.260,0.321


In [28]:
# ==========================================
# CELL 4: FF4 FACTOR LOADINGS (THE "BLACK BOX" EXPLAINER)
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Extracting Detailed Carhart 4-Factor Loadings...\n")

# Let's look closely at the Mutual Fund (Long-Only) and the Hedge Fund (Long-Short)
strategies_to_analyze = {
    'Modern Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Modern Ensemble (Long-Only)': 'Net_Ret_LongOnly'
}

for name, col in strategies_to_analyze.items():
    print(f"=== {name} FF4 Factor Loadings ===")
    
    # 1. Setup the Regression exactly like Cell 3
    Y = eval_df_mod[col] - eval_df_mod['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # 2. Extract the detailed statistics
    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    
    # 3. Add significance stars
    def get_stars(pval):
        if pval < 0.01: return '***'
        elif pval < 0.05: return '**'
        elif pval < 0.10: return '*'
        return ''
    
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    
    # 4. Clean up the row names for display
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']
    
    # Format the numbers
    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)
    
    print(loadings_df)
    print("\n")

Extracting Detailed Carhart 4-Factor Loadings...

=== Modern Ensemble (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value sig
Alpha (monthly)   0.037857   2.422050  2.191335e-02  **
Market (β_mkt)    0.787801   2.445706  2.076168e-02  **
SMB (β_smb)      -0.063959  -0.116836  9.077957e-01    
HML (β_hml)       0.745659   1.993509  5.568723e-02   *
UMD (β_umd)       0.691146   1.963668  5.922027e-02   *


=== Modern Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value sig
Alpha (monthly)   0.004613   0.407598  6.865621e-01    
Market (β_mkt)   -0.014490  -0.062121  9.508928e-01    
SMB (β_smb)       0.161753   0.408059  6.862274e-01    
HML (β_hml)      -0.501929  -1.853166  7.405895e-02   *
UMD (β_umd)      -0.189968  -0.745372  4.620458e-01    




### Historic evaluation

In [29]:
# ==========================================
# CELL 2: RISK METRICS (SHARPE & DRAWDOWN)
# ==========================================
import pandas as pd
import numpy as np

print("Calculating Risk Metrics...\n")

risk_results = []

for col in strategy_cols_hist:
    # 1. Calculate Excess Return for the strategy
    # Note: If the strategy is 'VW', it is already the market, but we still subtract Rf for its Sharpe
    excess_return = eval_df_hist[col] - eval_df_hist['Rf(1m)']
    
    # 2. Calculate Annualized Return and Volatility
    ann_ret = eval_df_hist[col].mean() * 12
    ann_vol = eval_df_hist[col].std() * np.sqrt(12)
    
    # 3. Calculate Annualized Sharpe Ratio
    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0
    
    # 4. Calculate Maximum Drawdown
    # Create a cumulative wealth index starting at 1.0
    cum_wealth = (1 + eval_df_hist[col]).cumprod()
    # Track the highest peak achieved so far
    running_max = cum_wealth.cummax()
    # Calculate the percentage drop from the peak
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()
    
    # Save the metrics
    risk_results.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

# Convert to a DataFrame for clean formatting
risk_df = pd.DataFrame(risk_results)

# Format the output beautifully for your thesis
risk_df['Ann_Return'] = risk_df['Ann_Return'].map('{:.2%}'.format)
risk_df['Ann_Volatility'] = risk_df['Ann_Volatility'].map('{:.2%}'.format)
risk_df['Sharpe_Ratio'] = risk_df['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df['Max_Drawdown'] = risk_df['Max_Drawdown'].map('{:.2%}'.format)

print("=== RISK-ADJUSTED PERFORMANCE METRICS ===")
print(risk_df)

Calculating Risk Metrics...

=== RISK-ADJUSTED PERFORMANCE METRICS ===
             Strategy Ann_Return Ann_Volatility Sharpe_Ratio Max_Drawdown
0    Net_Ret_LongOnly     24.50%         23.12%         1.02      -22.83%
1   Net_Ret_LongShort     30.54%         30.96%         0.96      -19.76%
2  Net_Ret_Hist_RF_LS     36.97%         25.58%         1.41      -15.47%
3      Net_Ret_Mom_LS     15.39%         29.56%         0.49      -24.05%
4                  VW     -0.19%         16.18%        -0.07      -27.04%


In [32]:
# ==========================================
# CELL 3: THE ULTIMATE ALPHA REGRESSIONS
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Running CAPM, FF3, and Carhart 4-Factor Regressions...\n")

# We don't need to run this on the VW market itself, just our strategies
strategies_to_test = [
    'Net_Ret_LongOnly', 
    'Net_Ret_LongShort', 
    'Net_Ret_Hist_RF_LS', 
    'Net_Ret_Mom_LS'
]

# Create an empty list to store our rows of data
alpha_results = []

for strat in strategies_to_test:
    # Our Dependent Variable (Y) is the Strategy's Excess Return
    Y = eval_df_hist[strat] - eval_df_hist['Rf(1m)']
    
    # ----------------------------------------------------
    # Model 1: CAPM (1-Factor)
    # ----------------------------------------------------
    X_capm = sm.add_constant(eval_df_hist[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()
    
    # ----------------------------------------------------
    # Model 2: Fama-French 3-Factor (FF3)
    # ----------------------------------------------------
    X_ff3 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()
    
    # ----------------------------------------------------
    # Model 3: Carhart 4-Factor (FF4)
    # ----------------------------------------------------
    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # Extract Alpha (Intercept) and annualize it (* 12)
    # We also extract the t-stat, p-value, and R-squared
    alpha_results.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],
        
        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],
        
        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

# Format the results into a beautiful Pandas DataFrame
alpha_df = pd.DataFrame(alpha_results)

# Create a mapping function to add significance stars based on p-values
def format_alpha(alpha, pval):
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f"{alpha:.2%}{stars}"

# Apply formatting
for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df[f'{model}_Alpha_Str'] = alpha_df.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df[f'{model}_t'] = alpha_df[f'{model}_t'].map('{:.2f}'.format)
    alpha_df[f'{model}_p'] = alpha_df[f'{model}_p'].map('{:.3f}'.format)

# Reorder columns to match your requested table format perfectly
final_table = alpha_df[[
    'Strategy', 
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]]

# Rename for display
final_table.columns = [
    'Strategy', 
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha', 
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat', 
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== COMBINED ALPHA SUMMARY TABLE ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
print(final_table)

Running CAPM, FF3, and Carhart 4-Factor Regressions...

=== COMBINED ALPHA SUMMARY TABLE ===
Note: *** p<0.01, ** p<0.05, * p<0.10

             Strategy CAPM Alpha FF3 Alpha FF4 Alpha CAPM t-stat FF3 t-stat  \
0    Net_Ret_LongOnly     23.52%    -4.03%    -2.06%        1.68      -0.29   
1   Net_Ret_LongShort     29.97%  45.25%**   41.71%*        1.63       2.10   
2  Net_Ret_Hist_RF_LS   36.34%**  45.33%**  41.65%**        2.40       2.56   
3      Net_Ret_Mom_LS     14.85%    23.93%    21.01%        0.85       1.15   

  FF4 t-stat CAPM p-val FF3 p-val FF4 p-val  
0      -0.15      0.102     0.775     0.884  
1       1.96      0.112     0.044     0.059  
2       2.43      0.022     0.016     0.022  
3       1.01      0.402     0.260     0.321  


In [34]:
# ==========================================
# CELL 4: FF4 FACTOR LOADINGS (THE "BLACK BOX" EXPLAINER)
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Extracting Detailed Carhart 4-Factor Loadings...\n")

# Let's look closely at the Mutual Fund (Long-Only) and the Hedge Fund (Long-Short)
strategies_to_analyze = {
    'Historical Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Historical Ensemble (Long-Only)': 'Net_Ret_LongOnly',
    'Historical RF (Long-Short)': 'Net_Ret_Hist_RF_LS',
}

for name, col in strategies_to_analyze.items():
    print(f"=== {name} FF4 Factor Loadings ===")
    
    # 1. Setup the Regression exactly like Cell 3
    Y = eval_df_hist[col] - eval_df_hist['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # 2. Extract the detailed statistics
    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    
    # 3. Add significance stars
    def get_stars(pval):
        if pval < 0.01: return '***'
        elif pval < 0.05: return '**'
        elif pval < 0.10: return '*'
        return ''
    
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    
    # 4. Clean up the row names for display
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']
    
    # Format the numbers
    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)
    
    print(loadings_df)
    print("\n")

Extracting Detailed Carhart 4-Factor Loadings...

=== Historical Ensemble (Long-Short) FF4 Factor Loadings ===
                     Coef    t-stat       p-value sig
Alpha (monthly)  0.034759  1.962223  5.939627e-02   *
Market (β_mkt)   0.514961  1.410591  1.690026e-01    
SMB (β_smb)      0.238266  0.384041  7.037498e-01    
HML (β_hml)      0.710381  1.675749  1.045380e-01    
UMD (β_umd)      0.579225  1.452061  1.572187e-01    


=== Historical Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value sig
Alpha (monthly)  -0.001719  -0.147486  8.837689e-01    
Market (β_mkt)   -0.219285  -0.913113  3.687125e-01    
SMB (β_smb)       0.365470   0.895481  3.779031e-01    
HML (β_hml)      -0.639074  -2.291698  2.937361e-02  **
UMD (β_umd)      -0.322595  -1.229376  2.288127e-01    


=== Historical RF (Long-Short) FF4 Factor Loadings ===
                     Coef    t-stat       p-value sig
Alpha (monthly)  0.034707  2.427984  2.161912e-02  **
Ma